In [1]:
using Pkg
Pkg.activate("Data_Assimilation")
Pkg.develop(path="../Krylov.jl")   # change path to local Krylov fork
Pkg.develop(path="../JSOSolvers.jl") 

  Activating project at `~/Desktop/DataAssim.jl/Data_Assimilation`
   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Manifest.toml`


In [2]:

# Inclure tes fichiers locaux
include("../DataAssim.jl/src/lorenz95.jl")
include("../DataAssim.jl/src/operators.jl")

   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Manifest.toml`


build_model (generic function with 1 method)

In [3]:
using NLPModels, ADNLPModels
using JSOSolvers

function nonlinear_funcval(x, y, obs, xb, B, R)
    eo = y .- gop(obs, x)
    eb = x .- xb
    return (1/2) * eb'* invdot(B, eb) + (1/2) * eo'*invdot(R, eo)
end

nonlinear_funcval (generic function with 1 method)

In [4]:
n = 1000
nt = 8
dt = 0.025
F = 8.0
rng = MersenneTwister(1234)

sigmaR =  0.1
total_space_obs = 300
total_time_obs = 2   # m = total_space_obs*total_time_obs
sigmaB =  0.8

# Model
model = Lorenz95Model(F, dt)

# Observations

space_inds_obs = round.(Int, range(1, n; length=total_space_obs))
time_inds_obs = round.(Int, range(1, nt-1; length=total_time_obs))
m = total_space_obs*total_time_obs
obs = ObsOperator(sigmaR, space_inds_obs, n, time_inds_obs, nt, m, model)
R = RMatrix(sigmaR)

# Background
B = BMatrix(sigmaB, n)

# Spin-up
xt = 3.0 .* ones(n) .+ randn(rng, n)
xt = traj(model, xt, 5000)

# Background state
xb = xt .+ randn(rng, n) .* sigmaB

# Observations
y = generate_obs(obs, xt)

# Construct ADNLPModel
f(x) = nonlinear_funcval(x, y, obs, xb, B, R)

f (generic function with 1 method)

In [5]:
x0 =  xb
nlp = ADNLPModel(f, x0)

ADNLPModel - Model with automatic differentiation backend ADModelBackend{
  ForwardDiffADGradient,
  ForwardDiffADHvprod,
  EmptyADbackend,
  EmptyADbackend,
  EmptyADbackend,
  SparseADHessian,
  EmptyADbackend,
}
  Problem name: Generic
   All variables: ████████████████████ 1000   All constraints: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
            free: ████████████████████ 1000              free: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           lower: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                lower: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           upper: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                upper: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
         low/upp: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0              low/upp: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           fixed: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                fixed: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
          infeas: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0               infeas: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
            nnzh: ( 83.16% sparsity)   84300           linear: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
                                 

In [7]:
Pkg.add("DataFrames")
using DataFrames
function run_solver(nlp, subsolver; memory=nothing)
    kwargs = memory === nothing ? () : (subsolver_kwargs=(memory=memory,),)

    stats = trunk(nlp,
        max_time=10000.0,
        max_iter=500,
        verbose=0,
        subsolver=subsolver;
        kwargs...
    )

    row = (
        solver = string(subsolver) * (memory === nothing ? "" : "_m$(memory)"),
        status = stats.status,
        norm_sol = norm(stats.solution),
        objective = stats.objective,
        iter = stats.iter,
        obj_eval = nlp.counters.neval_obj,
        grad_eval = nlp.counters.neval_grad,
        hprod = nlp.counters.neval_hprod,
        time = stats.elapsed_time
    )

    reset!(nlp)

    return row
end

   Resolving package versions...
    Updating `~/Desktop/DataAssim.jl/Data_Assimilation/Project.toml`
  [a93c6f00] + DataFrames v1.8.1
    Updating `~/Desktop/DataAssim.jl/Data_Assimilation/Manifest.toml`
  [a8cc5b0e] + Crayons v4.1.1
  [a93c6f00] + DataFrames v1.8.1
  [e2d170a0] + DataValueInterfaces v1.0.0
  [842dd82b] + InlineStrings v1.4.5
  [41ab1584] + InvertedIndices v1.3.1
  [82899510] + IteratorInterfaceExtensions v1.0.0
  [b964fa9f] + LaTeXStrings v1.4.0
  [2dfb63ee] + PooledArrays v1.4.3
  [08abe8d2] + PrettyTables v3.3.2
  [91c51154] + SentinelArrays v1.4.9
  [892a3eda] + StringManipulation v0.4.4
  [3783bdb8] + TableTraits v1.0.1
  [bd369af6] + Tables v1.12.1
  [9fa8497b] + Future v1.11.0
  [3fa0cd96] + REPL v1.11.0
  [6462fe0b] + Sockets v1.11.0
  [f489334b] + StyledStrings v1.11.0
Precompiling project...
  28680.3 ms  ✓ DataFrames
  1 dependency successfully precompiled in 30 seconds. 148 already precompiled.


run_solver (generic function with 1 method)

In [8]:
results = DataFrame()

push!(results, run_solver(nlp, :cg))
push!(results, run_solver(nlp, :lbfgs, memory=100))
push!(results, run_solver(nlp, :diom, memory=100))
push!(results, run_solver(nlp, :lbfgs, memory=50))
push!(results, run_solver(nlp, :diom, memory=50))

InterruptException: InterruptException:

In [ ]:
using PrettyTables
pretty_table(results)